# Hidden-Change Business Idea Finder — All-API `text-ingest` Notebook

This notebook uses [`Woys/text-ingest`](https://github.com/Woys/text-ingest) as the ingestion layer and turns cross-source industry signals into **business idea hypotheses**.

Default industry: **marketing agency**.

The goal is not just to find “news.” The goal is to detect **hidden industry changes** before they become obvious:

- policy/regulation changes that create compliance pain
- inventions, research, patents, and open-source tools that change what is possible
- platform/API/policy changes that break old workflows
- client budget, in-housing, and procurement shifts
- public-company filings that reveal strategic pressure
- developer/practitioner chatter that appears before mainstream adoption
- weak signals across Reddit, Hacker News, GitHub, Stack Exchange, blogs, research, filings, and news

Output files:

- `raw_hidden_change_records.jsonl`
- `clean_hidden_change_signals.csv`
- `hidden_change_clusters.csv`
- `business_idea_hypotheses.csv`
- `idea_validation_backlog.csv`
- `hidden_change_business_ideas_report.md`
- `api_fetcher_coverage.csv`

## 0. Setup

Run the next setup cell before the rest of the notebook. It fixes the most common issue shown by your screenshot: Jupyter imports an older `data_ingestion` package from `site-packages`, so only a few fetchers build.

The setup cell will:

1. Find a local `Woys/text-ingest` repo if you are already inside it.
2. Otherwise clone `https://github.com/Woys/text-ingest.git` into the current folder.
3. Put the repo's `src/` folder first on `sys.path`, so the notebook uses the newest repo code instead of the old installed package.
4. Try an editable install with dev dependencies.
5. Print where `data_ingestion` is being loaded from.

Optional API keys:

```bash
export NEWSAPI_KEY="..."
export GUARDIAN_API_KEY="..."
export GITHUB_TOKEN="..."
```

Without `NEWSAPI_KEY`, NewsAPI will be skipped because that fetcher requires a key. Most other sources can still run without paid keys.


In [ ]:
# Uncomment only if you are not already inside the repo.
# !git clone https://github.com/Woys/text-ingest.git
# %cd text-ingest

# Run once in a fresh environment.
# %pip install -e ".[dev]"
# %pip install pandas numpy matplotlib scikit-learn python-dateutil tqdm

In [ ]:
# Force the notebook to use the current GitHub repo code, not an old installed package.
# This fixes the issue where only openalex/crossref/federalregister/hackernews build.


import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/Woys/text-ingest.git"
REPO_DIR_NAME = "text-ingest"


def find_text_ingest_repo(start: Path | None = None) -> Path | None:
    """Find a local Woys/text-ingest repo by walking current/parent/common folders."""
    start = (start or Path.cwd()).resolve()
    candidates = []

    # Current directory and parents.
    for p in [start, *start.parents]:
        candidates.append(p)
        candidates.append(p / REPO_DIR_NAME)

    # Common notebook locations.
    candidates.extend(
        [
            Path.home() / REPO_DIR_NAME,
            Path.cwd() / REPO_DIR_NAME,
            Path("/content") / REPO_DIR_NAME,
            Path("/mnt/data") / REPO_DIR_NAME,
        ]
    )

    seen = set()
    for candidate in candidates:
        candidate = candidate.resolve()
        if candidate in seen:
            continue
        seen.add(candidate)
        if (candidate / "src" / "data_ingestion" / "config.py").exists():
            return candidate
    return None


repo_root = find_text_ingest_repo()

if repo_root is None:
    repo_root = (Path.cwd() / REPO_DIR_NAME).resolve()
    if not repo_root.exists():
        print(f"Cloning {REPO_URL} into {repo_root} ...")
        subprocess.run(["git", "clone", REPO_URL, str(repo_root)], check=True)
    else:
        print(
            f"Found folder {repo_root}, but it does not look like the repo. Check path if this fails."
        )
else:
    print(f"Found text-ingest repo: {repo_root}")

# Pull latest if it is a git repo. Non-fatal if offline or local changes exist.
if (repo_root / ".git").exists():
    try:
        subprocess.run(["git", "-C", str(repo_root), "pull", "--ff-only"], check=False)
    except Exception as exc:
        print("Git pull skipped:", exc)

src_dir = repo_root / "src"
if not src_dir.exists():
    raise RuntimeError(f"Could not find repo src directory: {src_dir}")

# Make local src win over old site-packages.
sys.path = [p for p in sys.path if str(src_dir) != p]
sys.path.insert(0, str(src_dir))

# Remove previously imported old modules if the cell is rerun.
for module_name in list(sys.modules):
    if module_name == "data_ingestion" or module_name.startswith("data_ingestion."):
        del sys.modules[module_name]

# Editable install makes console/dependencies consistent. Non-fatal because sys.path already points at src/.
try:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-e", str(repo_root) + "[dev]"],
        check=False,
    )
except Exception as exc:
    print("Editable install skipped:", exc)

import data_ingestion

print("Using data_ingestion from:", data_ingestion.__file__)
print("Repo root:", repo_root)
print("src first on sys.path:", sys.path[0])
print(
    "\nIf this path still points to site-packages, restart the kernel and run this setup cell first."
)

## 1. Configure the target industry

Edit this cell to switch from marketing agencies to another industry.

The most important variables are:

- `INDUSTRY`
- `INDUSTRY_TERMS`
- `ADJACENT_TERMS`
- `CUSTOMER_TYPES`
- `WATCH_SITES`

In [ ]:
import json
import math
import re
from collections import Counter
from datetime import date, datetime, timedelta
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd

# -----------------------------------------------------------------------------
# TARGET INDUSTRY
# -----------------------------------------------------------------------------
INDUSTRY = "marketing agency"

INDUSTRY_DESCRIPTION = (
    "Agencies and consultancies that sell digital marketing, paid media, creative, "
    "SEO/SEM, social, influencer, content, analytics, CRO, marketing automation, "
    "brand strategy, and performance marketing services to clients."
)

CUSTOMER_TYPES = [
    "independent marketing agencies",
    "performance marketing agencies",
    "creative agencies",
    "media buying teams",
    "agency owners",
    "agency operations leaders",
    "agency analytics teams",
    "CMOs at mid-market brands",
    "growth teams at ecommerce companies",
]

INDUSTRY_TERMS = [
    "marketing agency",
    "advertising agency",
    "digital agency",
    "creative agency",
    "media agency",
    "performance marketing",
    "digital marketing",
    "growth marketing",
    "paid media",
    "media buying",
    "programmatic advertising",
    "search advertising",
    "paid search",
    "paid social",
    "social media advertising",
    "PPC",
    "SEM",
    "SEO",
    "content marketing",
    "email marketing",
    "influencer marketing",
    "creator marketing",
    "brand strategy",
    "conversion rate optimization",
    "CRO",
    "marketing analytics",
    "attribution",
    "lead generation",
    "demand generation",
    "account-based marketing",
    "agency retainer",
]

ADJACENT_TERMS = [
    "AI advertising",
    "generative AI",
    "AI agent",
    "agentic",
    "synthetic media",
    "AI disclosure",
    "deepfake",
    "virtual influencer",
    "adtech",
    "martech",
    "first-party data",
    "third-party cookies",
    "cookie deprecation",
    "privacy",
    "consent management",
    "data clean room",
    "conversion API",
    "incrementality",
    "marketing mix modeling",
    "MMM",
    "brand safety",
    "retail media",
    "connected TV",
    "CTV",
    "algorithm change",
    "platform policy",
    "Google Ads",
    "Meta Ads",
    "TikTok Ads",
    "LinkedIn Ads",
    "Amazon Ads",
    "CRM",
    "CDP",
    "GA4",
    "FTC",
    "IAB",
    "NAD",
    "claim substantiation",
]

# Hidden-change trigger terms: language that often indicates a new pain, new constraint, or new enabling technology.
HIDDEN_CHANGE_TRIGGER_TERMS = [
    "proposed rule",
    "final rule",
    "guidance",
    "enforcement",
    "settlement",
    "fine",
    "penalty",
    "lawsuit",
    "class action",
    "investigation",
    "compliance",
    "disclosure",
    "privacy",
    "consent",
    "deprecation",
    "sunset",
    "phase out",
    "ban",
    "restriction",
    "policy update",
    "API change",
    "algorithm change",
    "account suspension",
    "measurement loss",
    "signal loss",
    "attribution gap",
    "new standard",
    "framework",
    "launch",
    "rollout",
    "beta",
    "patent",
    "research",
    "study",
    "open source",
    "repository",
    "automation",
    "AI agent",
    "generative AI",
    "agentic",
    "synthetic media",
    "in-housing",
    "self-serve",
    "budget cut",
    "fee pressure",
    "pricing pressure",
    "margin pressure",
    "client churn",
    "procurement",
    "RFP",
    "consolidation",
    "merger",
    "acquisition",
    "layoff",
    "reskilling",
]

PAIN_TERMS = [
    "problem",
    "pain",
    "challenge",
    "headwind",
    "barrier",
    "constraint",
    "bottleneck",
    "manual",
    "expensive",
    "slow",
    "complex",
    "fragmented",
    "inaccurate",
    "unreliable",
    "hard to measure",
    "compliance burden",
    "risk",
    "uncertainty",
    "loss",
    "decline",
    "gap",
    "shortage",
    "delay",
]

OPPORTUNITY_TERMS = [
    "growth",
    "increase",
    "funding",
    "investment",
    "partnership",
    "contract",
    "new platform",
    "standard",
    "framework",
    "launch",
    "adoption",
    "breakthrough",
    "patent",
    "open source",
    "automation",
    "productivity",
    "personalization",
    "optimization",
    "first-party data",
    "clean room",
    "incrementality",
    "retail media",
]

# Dates: a shorter window is useful for daily monitoring; a longer window is useful for idea discovery.
END_DATE = date.today().isoformat()
START_DATE = (date.today() - timedelta(days=180)).isoformat()
MAX_PAGES = 1  # Increase after you verify API keys and rate limits.

OUT_DIR = Path("data/hidden_change_business_ideas_marketing_agency")
OUT_DIR.mkdir(parents=True, exist_ok=True)

RAW_JSONL = OUT_DIR / "raw_hidden_change_records.jsonl"
CLEAN_CSV = OUT_DIR / "clean_hidden_change_signals.csv"
CLUSTERS_CSV = OUT_DIR / "hidden_change_clusters.csv"
IDEAS_CSV = OUT_DIR / "business_idea_hypotheses.csv"
VALIDATION_BACKLOG_CSV = OUT_DIR / "idea_validation_backlog.csv"
REPORT_MD = OUT_DIR / "hidden_change_business_ideas_report.md"
API_COVERAGE_CSV = OUT_DIR / "api_fetcher_coverage.csv"

print(
    {
        "industry": INDUSTRY,
        "window": f"{START_DATE} to {END_DATE}",
        "output_dir": str(OUT_DIR),
    }
)

## 2. Build fetcher specs for all APIs/sources

The public repo currently lists these fetchers:

`openalex`, `crossref`, `newsapi`, `hackernews`, `federalregister`, `website`, `website_html`, `wikipedia`, `reddit`, `edgar`, `github`, `stackexchange`, `openlibrary`, `googlenews`, and `guardian`.

This notebook intentionally creates specs for all of them. Unsupported or misconfigured sources are skipped safely during execution.

In [ ]:
def q(*parts: str) -> str:
    return " ".join(p.strip() for p in parts if p and p.strip())


QUERY_LENSES = {
    "general_industry": q(
        INDUSTRY,
        "business model client demand competition agency economics AI privacy platform changes",
    ),
    "policy_regulation": q(
        INDUSTRY,
        "FTC advertising law disclosure endorsement AI claims privacy consent synthetic media compliance NAD",
    ),
    "technology_inventions": q(
        INDUSTRY,
        "generative AI marketing agent martech adtech attribution measurement automation optimization patent research",
    ),
    "client_budget_shift": q(
        INDUSTRY,
        "client in-housing budget cut procurement pressure RFP retainer margin pricing agency fees",
    ),
    "platform_ops_shift": q(
        INDUSTRY,
        "Google Ads Meta Ads TikTok Ads Amazon Ads LinkedIn Ads platform policy API attribution tracking conversion",
    ),
    "measurement_data_shift": q(
        INDUSTRY,
        "cookie deprecation first-party data clean room consent mode GA4 attribution incrementality marketing mix modeling",
    ),
    "legal_reputation_shift": q(
        INDUSTRY,
        "deceptive advertising fine lawsuit class action privacy breach brand safety fake reviews dark patterns",
    ),
    "developer_practitioner_weak_signals": q(
        INDUSTRY,
        "open source tool API workflow automation attribution ads analytics marketing",
    ),
    "books_reports": q(
        INDUSTRY,
        "advertising marketing strategy agency management AI transformation privacy analytics",
    ),
}

ALL_REPO_FETCHER_SOURCES = [
    "openalex",
    "crossref",
    "newsapi",
    "hackernews",
    "federalregister",
    "website",
    "website_html",
    "wikipedia",
    "reddit",
    "edgar",
    "github",
    "stackexchange",
    "openlibrary",
    "googlenews",
    "guardian",
]

COMMON_CONFIG = {
    "max_pages": MAX_PAGES,
    "start_date": START_DATE,
    "end_date": END_DATE,
    "languages": ["en"],
    "topic_include": INDUSTRY_TERMS + ADJACENT_TERMS + HIDDEN_CHANGE_TRIGGER_TERMS,
}


def cfg(query: str, **overrides: Any) -> dict[str, Any]:
    base = {**COMMON_CONFIG, "query": query}
    base.update({k: v for k, v in overrides.items() if v is not None})
    return base


def website_cfg(
    site_url: str, query: str | None = None, **overrides: Any
) -> dict[str, Any]:
    base = {
        "site_url": site_url,
        "query": query or INDUSTRY,
        "max_items": 50,
        "start_date": START_DATE,
        "end_date": END_DATE,
        "languages": ["en"],
    }
    base.update({k: v for k, v in overrides.items() if v is not None})
    return base


def website_html_cfg(
    site_url: str, query: str | None = None, **overrides: Any
) -> dict[str, Any]:
    base = {
        "site_url": site_url,
        "query": query or INDUSTRY,
        "max_items": 25,
        "max_candidate_links": 300,
        "include_list_pages_as_items": True,
        "start_date": START_DATE,
        "end_date": END_DATE,
        "languages": ["en"],
        "link_include_patterns": [
            "blog",
            "news",
            "guidance",
            "guidelines",
            "standards",
            "policy",
            "privacy",
            "advertising",
            "marketing",
            "ads",
            "ai",
            "measurement",
            "analytics",
            "commerce",
            "platform",
            "api",
        ],
        "link_exclude_patterns": [
            "login",
            "signup",
            "subscribe",
            "careers",
            "privacy-policy",
            "terms",
            "contact",
        ],
    }
    base.update({k: v for k, v in overrides.items() if v is not None})
    return base


# Agency-specific watched sources. Add/remove sources depending on the vertical.
WATCH_SITES = [
    {
        "name": "FTC Business Guidance Blog",
        "site_url": "https://www.ftc.gov/business-guidance/blog",
    },
    {"name": "FTC News", "site_url": "https://www.ftc.gov/news-events/news"},
    {"name": "IAB News", "site_url": "https://www.iab.com/news/"},
    {"name": "IAB Guidelines", "site_url": "https://www.iab.com/guidelines/"},
    {"name": "IAB Tech Lab Standards", "site_url": "https://iabtechlab.com/standards/"},
    {
        "name": "Google Ads & Commerce Blog",
        "site_url": "https://blog.google/products/ads-commerce/",
    },
    {
        "name": "Meta for Business News",
        "site_url": "https://www.facebook.com/business/news",
    },
    {"name": "Think with Google", "site_url": "https://www.thinkwithgoogle.com/"},
    {
        "name": "TikTok Business Blog",
        "site_url": "https://www.tiktok.com/business/en/blog",
    },
    {
        "name": "Amazon Ads Blog",
        "site_url": "https://advertising.amazon.com/library/guides",
    },
    {
        "name": "LinkedIn Marketing Blog",
        "site_url": "https://www.linkedin.com/business/marketing/blog",
    },
    {"name": "Google Privacy Sandbox", "site_url": "https://privacysandbox.com/news/"},
]

FETCHER_SPECS: list[dict[str, Any]] = []

# Research / invention signals.
FETCHER_SPECS += [
    {
        "source": "openalex",
        "config": cfg(QUERY_LENSES["technology_inventions"], per_page=50),
    },
    {
        "source": "crossref",
        "config": cfg(QUERY_LENSES["technology_inventions"], rows=50),
    },
    {
        "source": "openlibrary",
        "config": cfg(QUERY_LENSES["books_reports"], page_size=50),
    },
]

# Policy and regulation.
FETCHER_SPECS += [
    {
        "source": "federalregister",
        "config": cfg(QUERY_LENSES["policy_regulation"], per_page=50),
    },
]

# News and market signals.
FETCHER_SPECS += [
    {
        "source": "googlenews",
        "config": cfg(
            QUERY_LENSES["general_industry"],
            page_size=50,
            hl="en-US",
            gl="US",
            ceid="US:en",
        ),
    },
    {
        "source": "guardian",
        "config": cfg(
            QUERY_LENSES["general_industry"],
            page_size=50,
            api_key=os.getenv("GUARDIAN_API_KEY"),
        ),
    },
    {
        "source": "newsapi",
        "config": cfg(
            QUERY_LENSES["general_industry"],
            page_size=50,
            api_key=os.getenv("NEWSAPI_KEY"),
            language="en",
        ),
    },
]

# Filings and competition.
FETCHER_SPECS += [
    {
        "source": "edgar",
        "config": cfg(QUERY_LENSES["client_budget_shift"], per_page=50),
    },
]

# Developer/practitioner weak signals.
FETCHER_SPECS += [
    {
        "source": "github",
        "config": cfg(
            QUERY_LENSES["developer_practitioner_weak_signals"],
            per_page=50,
            sort="updated",
            github_token=os.getenv("GITHUB_TOKEN"),
        ),
    },
    {
        "source": "hackernews",
        "config": cfg(
            QUERY_LENSES["developer_practitioner_weak_signals"],
            hits_per_page=50,
            use_date_sort=True,
        ),
    },
    {
        "source": "reddit",
        "config": cfg(QUERY_LENSES["client_budget_shift"], page_size=50, sort="new"),
    },
    {
        "source": "stackexchange",
        "config": cfg(
            QUERY_LENSES["developer_practitioner_weak_signals"],
            page_size=50,
            site="stackoverflow",
            sort="activity",
        ),
    },
]

# Reference/background.
FETCHER_SPECS += [
    {
        "source": "wikipedia",
        "config": cfg(
            q(
                INDUSTRY,
                "advertising technology marketing automation privacy artificial intelligence",
            ),
            page_size=25,
        ),
    },
]

# Watched sites: try both RSS/autodiscovery and HTML fallback.
for site in WATCH_SITES:
    FETCHER_SPECS.append(
        {
            "source": "website",
            "config": website_cfg(
                site["site_url"], query=QUERY_LENSES["platform_ops_shift"]
            ),
        }
    )
    FETCHER_SPECS.append(
        {
            "source": "website_html",
            "config": website_html_cfg(
                site["site_url"], query=QUERY_LENSES["platform_ops_shift"]
            ),
        }
    )

spec_preview = pd.DataFrame(
    [
        {
            "source": spec["source"],
            "query_or_site": spec["config"].get("query")
            or spec["config"].get("site_url"),
            "site_url": spec["config"].get("site_url", ""),
        }
        for spec in FETCHER_SPECS
    ]
)
print(
    f"Prepared {len(FETCHER_SPECS)} fetcher specs across {spec_preview['source'].nunique()} source types."
)
spec_preview.head(30)

## 3. Run ingestion safely

This cell tries to build and run all fetchers. It does **not** crash the whole notebook when one API is unsupported, missing a key, or rate-limited.

In [ ]:
from typing import Literal, get_args, get_origin

# This import should come from the repo src/ path printed by the setup cell, not old site-packages.
from data_ingestion.config import (
    FetcherSpec,
    JsonlSinkConfig,
    PipelineConfig,
    RuntimeOptimizationConfig,
)
from data_ingestion.factories import build_fetcher
from data_ingestion.pipeline import DataDumperPipeline
from data_ingestion.sinks.jsonl import JsonlSink


def _literal_values(annotation: Any) -> set[str]:
    """Extract string Literal values from a possibly nested Pydantic/typing annotation."""
    origin = get_origin(annotation)
    if origin is Literal:
        return {x for x in get_args(annotation) if isinstance(x, str)}
    values: set[str] = set()
    for arg in get_args(annotation):
        values |= _literal_values(arg)
    return values


def allowed_fetcher_sources() -> set[str] | None:
    """Return sources allowed by the installed package's FetcherSpec, if introspection works."""
    try:
        source_field = FetcherSpec.model_fields["source"]
        values = _literal_values(source_field.annotation)
        return values or None
    except Exception:
        return None


allowed_sources = allowed_fetcher_sources()
if allowed_sources:
    print("Currently loaded package supports:", sorted(allowed_sources))
    if len(allowed_sources) < len(ALL_REPO_FETCHER_SOURCES):
        missing = sorted(set(ALL_REPO_FETCHER_SOURCES) - set(allowed_sources))
        print(
            "WARNING: your active data_ingestion package still supports fewer fetchers than the GitHub repo."
        )
        print("Missing fetchers:", missing)
        print(
            "Fix: restart kernel, run the setup cell first, and confirm data_ingestion.__file__ points to the repo src/ folder."
        )
else:
    print(
        "Could not introspect installed fetcher sources; will attempt all specs one by one."
    )

fetchers = []
coverage_rows = []

for i, spec in enumerate(FETCHER_SPECS, start=1):
    source = spec.get("source")
    status = "built"
    reason = ""
    if allowed_sources is not None and source not in allowed_sources:
        status = "skipped"
        reason = "not supported by installed package; update repo and reinstall with pip install -e '.[dev]'"
    else:
        try:
            fetchers.append(build_fetcher(spec))
        except Exception as exc:
            status = "skipped"
            reason = f"{type(exc).__name__}: {str(exc)[:300]}"
    coverage_rows.append(
        {
            "idx": i,
            "source": source,
            "status": status,
            "reason": reason,
            "query": spec.get("config", {}).get("query", ""),
            "site_url": spec.get("config", {}).get("site_url", ""),
        }
    )

coverage_df = pd.DataFrame(coverage_rows)
coverage_df.to_csv(API_COVERAGE_CSV, index=False)
print(coverage_df["status"].value_counts(dropna=False).to_dict())
display(coverage_df)

if fetchers:
    pipeline_config = PipelineConfig(
        fail_fast=False,
        runtime=RuntimeOptimizationConfig(
            enable_deduplication=True,
            enable_quality_checks=True,
            write_raw_payload=False,
        ),
    )
    sink = JsonlSink(JsonlSinkConfig(output_file=str(RAW_JSONL), append=False))
    pipeline = DataDumperPipeline(sink=sink, config=pipeline_config)
    summary = pipeline.run(fetchers)
    print(summary.model_dump() if hasattr(summary, "model_dump") else summary)
else:
    print(
        "No fetchers were built. Update the package, check API keys, or inspect api_fetcher_coverage.csv."
    )

### If you still see many skipped APIs

Check the printed line `Using data_ingestion from:` in the setup cell.

Good path example:

```text
.../text-ingest/src/data_ingestion/__init__.py
```

Bad path example:

```text
.../site-packages/data_ingestion/__init__.py
```

If you see `site-packages`, restart the kernel and run the setup cell first. Your screenshot showed the old-package symptom: only the older sources were available, while newer repo fetchers like `edgar`, `github`, `googlenews`, `guardian`, `reddit`, `website`, and `website_html` were not registered.


## 4. Load, normalize, and deduplicate records

In [ ]:
def read_jsonl(path: Path) -> pd.DataFrame:
    rows = []
    if not path.exists():
        return pd.DataFrame()
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError:
                continue
    return pd.DataFrame(rows)


def norm_text(value: Any) -> str:
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return ""
    if isinstance(value, list):
        return " ".join(norm_text(v) for v in value)
    return str(value)


def compact_spaces(text: str) -> str:
    return re.sub(r"\s+", " ", str(text)).strip()


def normalize_title(title: str) -> str:
    title = title.lower()
    title = re.sub(r"[^a-z0-9]+", " ", title)
    return compact_spaces(title)


raw_df = read_jsonl(RAW_JSONL)
print(f"Raw rows: {len(raw_df):,}")

for col in [
    "source",
    "external_id",
    "title",
    "abstract",
    "full_text",
    "url",
    "topic",
    "record_type",
    "published_date",
    "fetched_at",
]:
    if col not in raw_df.columns:
        raw_df[col] = None

if raw_df.empty:
    df = pd.DataFrame(
        columns=[
            "source",
            "title",
            "abstract",
            "full_text",
            "url",
            "topic",
            "record_type",
            "event_date",
            "age_days",
            "text",
        ]
    )
    print(
        "No records found. Continue after fixing API coverage, or widen queries/date window."
    )
else:
    for col in ["title", "abstract", "full_text", "url", "topic"]:
        raw_df[col] = raw_df[col].map(norm_text).map(compact_spaces)

    raw_df["text"] = (
        raw_df["title"].fillna("")
        + "\n"
        + raw_df["abstract"].fillna("")
        + "\n"
        + raw_df["full_text"].fillna("")
        + "\n"
        + raw_df["topic"].fillna("")
    ).map(compact_spaces)

    raw_df["published_date"] = pd.to_datetime(
        raw_df["published_date"], errors="coerce", utc=True
    )
    raw_df["fetched_at"] = pd.to_datetime(
        raw_df["fetched_at"], errors="coerce", utc=True
    )
    now_utc = pd.Timestamp.now(tz="UTC")
    raw_df["event_date"] = (
        raw_df["published_date"].fillna(raw_df["fetched_at"]).fillna(now_utc)
    )
    raw_df["age_days"] = (now_utc - raw_df["event_date"]).dt.days.clip(lower=0)

    raw_df["normalized_title"] = raw_df["title"].map(normalize_title)
    raw_df["dedupe_key"] = np.where(
        raw_df["url"].astype(bool),
        raw_df["url"].str.lower(),
        raw_df["source"].astype(str) + ":" + raw_df["normalized_title"],
    )
    df = raw_df.sort_values(
        ["event_date", "source"], ascending=[False, True]
    ).drop_duplicates("dedupe_key")
    df = df[df["text"].str.len() > 0].reset_index(drop=True)

print(f"After dedupe: {len(df):,}")
df[["source", "event_date", "title", "url"]].head(10)

## 5. Detect hidden-change categories

A hidden change is a non-obvious shift in constraints, capabilities, demand, or distribution that can create a new business opportunity.

In [ ]:
HIDDEN_CHANGE_TAXONOMY: dict[str, list[str]] = {
    "regulatory_friction_shift": [
        "regulation",
        "regulatory",
        "proposed rule",
        "final rule",
        "guidance",
        "enforcement",
        "fine",
        "penalty",
        "FTC",
        "Federal Trade Commission",
        "NAD",
        "endorsement",
        "disclosure",
        "claim substantiation",
        "deceptive advertising",
        "privacy",
        "consent",
        "state privacy",
        "children's privacy",
        "dark pattern",
        "fake review",
        "synthetic media",
        "AI disclosure",
        "deepfake",
        "consumer protection",
    ],
    "measurement_data_shift": [
        "attribution",
        "multi-touch attribution",
        "marketing mix modeling",
        "MMM",
        "incrementality",
        "lift test",
        "conversion tracking",
        "signal loss",
        "measurement loss",
        "third-party cookie",
        "cookie deprecation",
        "first-party data",
        "identity resolution",
        "clean room",
        "GA4",
        "server-side",
        "consent mode",
        "privacy sandbox",
    ],
    "ai_capability_shift": [
        "generative AI",
        "AI agent",
        "agentic",
        "automation",
        "creative automation",
        "dynamic creative",
        "synthetic media",
        "virtual influencer",
        "personalization",
        "optimization",
        "large language model",
        "LLM",
        "workflow automation",
        "prompt",
        "model",
        "hallucination",
        "human review",
        "brand voice",
    ],
    "platform_rule_distribution_shift": [
        "Google Ads",
        "Meta Ads",
        "TikTok Ads",
        "LinkedIn Ads",
        "Amazon Ads",
        "retail media",
        "programmatic",
        "DSP",
        "SSP",
        "algorithm change",
        "platform policy",
        "API change",
        "account suspension",
        "ad account",
        "brand safety",
        "CPM",
        "CPC",
        "ROAS",
        "CTV",
        "connected TV",
        "creator economy",
    ],
    "client_budget_procurement_shift": [
        "client",
        "budget",
        "budget cut",
        "budget freeze",
        "RFP",
        "pitch",
        "procurement",
        "retainer",
        "fee pressure",
        "pricing pressure",
        "margin pressure",
        "project-based",
        "performance-based compensation",
        "in-housing",
        "self-serve",
        "client churn",
        "vendor consolidation",
        "agency review",
    ],
    "competitive_structure_shift": [
        "competition",
        "competitor",
        "merger",
        "acquisition",
        "consolidation",
        "holding company",
        "consultancy",
        "WPP",
        "Publicis",
        "Omnicom",
        "Interpublic",
        "IPG",
        "Dentsu",
        "Accenture Song",
        "market share",
        "independent agency",
        "platform vendor",
        "retail media network",
    ],
    "new_tooling_supply_shift": [
        "open source",
        "github",
        "repository",
        "library",
        "API",
        "SDK",
        "software",
        "tool",
        "framework",
        "martech",
        "adtech",
        "customer data platform",
        "CDP",
        "CRM integration",
        "server-side tracking",
        "dashboard",
        "data quality",
        "analytics",
        "tagging",
        "implementation",
    ],
    "legal_trust_reputation_shift": [
        "lawsuit",
        "litigation",
        "settlement",
        "class action",
        "complaint",
        "investigation",
        "fraud",
        "controversy",
        "public backlash",
        "brand safety",
        "misinformation",
        "privacy",
        "security incident",
        "breach",
        "fake review",
        "dark pattern",
        "misleading",
        "rights of publicity",
    ],
    "talent_workflow_shift": [
        "staffing",
        "hiring",
        "layoff",
        "utilization",
        "billable",
        "billable hours",
        "resourcing",
        "account management",
        "project management",
        "scope creep",
        "freelancer",
        "outsourcing",
        "training",
        "reskilling",
        "workflow",
        "quality assurance",
        "QA",
        "creative review",
        "burnout",
        "capacity constraint",
    ],
    "macro_ad_spend_shift": [
        "inflation",
        "interest rate",
        "GDP",
        "recession",
        "unemployment",
        "geopolitical",
        "consumer confidence",
        "ad spend",
        "marketing budget",
        "economic uncertainty",
        "slowdown",
        "volatility",
    ],
}

SOURCE_HIDDENNESS_WEIGHT = {
    "federalregister": 1.00,  # regulatory changes before businesses react
    "edgar": 0.90,  # filings reveal strategy/risk before blog posts
    "github": 0.90,  # emerging tools before vendors package them
    "openalex": 0.85,  # research before commercialization
    "crossref": 0.85,
    "hackernews": 0.80,  # practitioner chatter
    "reddit": 0.75,
    "stackexchange": 0.75,
    "website_html": 0.70,
    "website": 0.70,
    "guardian": 0.60,
    "googlenews": 0.55,
    "newsapi": 0.55,
    "openlibrary": 0.45,
    "wikipedia": 0.25,
}


def find_terms(text: str, terms: list[str]) -> list[str]:
    text_l = str(text).lower()
    found = []
    for term in terms:
        t = term.lower()
        if re.search(r"(?<![a-z0-9])" + re.escape(t) + r"(?![a-z0-9])", text_l):
            found.append(term)
    return sorted(set(found), key=lambda x: x.lower())


def classify_many(text: str, taxonomy: dict[str, list[str]]) -> dict[str, list[str]]:
    return {
        label: find_terms(text, terms)
        for label, terms in taxonomy.items()
        if find_terms(text, terms)
    }


def primary_label(
    match_dict: dict[str, list[str]], default: str = "general_shift"
) -> str:
    if not match_dict:
        return default
    return max(match_dict.items(), key=lambda kv: len(kv[1]))[0]


def minmax(series: pd.Series) -> pd.Series:
    if len(series) == 0:
        return series
    s = series.fillna(0).astype(float)
    if s.max() == s.min():
        return pd.Series(np.zeros(len(s)), index=s.index)
    return (s - s.min()) / (s.max() - s.min())


if df.empty:
    scored_df = df.copy()
else:
    df["matched_industry_terms"] = df["text"].map(
        lambda x: find_terms(x, INDUSTRY_TERMS + ADJACENT_TERMS)
    )
    df["hidden_trigger_terms"] = df["text"].map(
        lambda x: find_terms(x, HIDDEN_CHANGE_TRIGGER_TERMS)
    )
    df["pain_terms"] = df["text"].map(lambda x: find_terms(x, PAIN_TERMS))
    df["opportunity_terms"] = df["text"].map(lambda x: find_terms(x, OPPORTUNITY_TERMS))
    df["hidden_change_matches"] = df["text"].map(
        lambda x: classify_many(x, HIDDEN_CHANGE_TAXONOMY)
    )
    df["hidden_change_category"] = df["hidden_change_matches"].map(primary_label)
    df["hidden_change_categories"] = df["hidden_change_matches"].map(
        lambda d: list(d.keys())
    )
    df["source_hiddenness"] = df["source"].map(
        lambda s: SOURCE_HIDDENNESS_WEIGHT.get(str(s).lower(), 0.50)
    )

    df["industry_term_count"] = df["matched_industry_terms"].map(len)
    df["trigger_term_count"] = df["hidden_trigger_terms"].map(len)
    df["pain_term_count"] = df["pain_terms"].map(len)
    df["opportunity_term_count"] = df["opportunity_terms"].map(len)
    df["category_count"] = df["hidden_change_categories"].map(len)
    df["freshness_score"] = np.exp(-df["age_days"].fillna(365).clip(lower=0) / 120.0)

    df["relevance_score"] = minmax(df["industry_term_count"] * 1.5)
    df["change_language_score"] = minmax(
        df["trigger_term_count"] * 1.5
        + df["pain_term_count"] * 1.0
        + df["opportunity_term_count"] * 0.8
    )
    df["cross_domain_score"] = minmax(df["category_count"])

    df["hidden_change_score"] = (
        100
        * (
            0.25 * df["relevance_score"]
            + 0.30 * df["change_language_score"]
            + 0.20 * df["source_hiddenness"]
            + 0.15 * df["freshness_score"]
            + 0.10 * df["cross_domain_score"]
        )
    ).round(2)

    df["business_idea_seed_score"] = (
        100
        * (
            0.30 * df["hidden_change_score"] / 100
            + 0.25 * minmax(df["pain_term_count"])
            + 0.20 * minmax(df["opportunity_term_count"])
            + 0.15 * df["source_hiddenness"]
            + 0.10 * df["freshness_score"]
        )
    ).round(2)
    scored_df = df.sort_values(
        ["business_idea_seed_score", "hidden_change_score"], ascending=False
    ).reset_index(drop=True)

scored_df[
    [
        "source",
        "event_date",
        "hidden_change_category",
        "hidden_change_score",
        "business_idea_seed_score",
        "title",
        "url",
    ]
].head(25)

## 6. Cluster signals into hidden-change themes

Clustering helps merge scattered signals into one opportunity theme. The notebook uses TF-IDF + KMeans if scikit-learn is installed, with a fallback to taxonomy-only grouping.

In [ ]:
def simple_keywords(texts: list[str], top_n: int = 8) -> str:
    stop = set(
        [
            "the",
            "a",
            "an",
            "and",
            "or",
            "of",
            "for",
            "to",
            "in",
            "on",
            "with",
            "by",
            "from",
            "as",
            "is",
            "are",
            "was",
            "were",
            "be",
            "been",
            "this",
            "that",
            "it",
            "its",
            "into",
            "about",
            "via",
            "at",
            "new",
            "using",
            "use",
            "more",
            "most",
            "can",
            "will",
            "may",
            "not",
            "but",
            "than",
            "their",
            "they",
            "them",
            "our",
            "your",
            "you",
            "we",
        ]
    )
    words = []
    for t in texts:
        words.extend(re.findall(r"[a-z][a-z0-9\-]{2,}", str(t).lower()))
    counts = Counter(w for w in words if w not in stop and len(w) > 2)
    return ", ".join([w for w, _ in counts.most_common(top_n)])


cluster_df = scored_df.copy()
cluster_df["cluster"] = -1
cluster_df["cluster_label"] = cluster_df.get("hidden_change_category", "general_shift")

if not cluster_df.empty:
    try:
        from sklearn.cluster import KMeans
        from sklearn.feature_extraction.text import TfidfVectorizer

        texts = cluster_df["text"].fillna("").astype(str)
        valid = texts[texts.str.len() > 30]
        if len(valid) >= 8:
            n_clusters = min(10, max(2, int(math.sqrt(len(valid)))))
            vectorizer = TfidfVectorizer(
                stop_words="english", max_features=2500, ngram_range=(1, 2), min_df=1
            )
            X = vectorizer.fit_transform(valid)
            kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init="auto")
            labels = kmeans.fit_predict(X)
            cluster_df.loc[valid.index, "cluster"] = labels
            features = np.array(vectorizer.get_feature_names_out())
            label_map = {}
            for cid in sorted(set(labels)):
                center = kmeans.cluster_centers_[cid]
                terms = features[np.argsort(center)[-7:]][::-1]
                label_map[int(cid)] = ", ".join(terms)
            cluster_df["cluster_label"] = (
                cluster_df["cluster"]
                .map(label_map)
                .fillna(cluster_df["hidden_change_category"])
            )
        else:
            cluster_df["cluster"] = pd.factorize(cluster_df["hidden_change_category"])[
                0
            ]
            cluster_df["cluster_label"] = cluster_df["hidden_change_category"]
    except Exception as exc:
        print(f"Clustering fallback used: {type(exc).__name__}: {exc}")
        cluster_df["cluster"] = pd.factorize(cluster_df["hidden_change_category"])[0]
        cluster_df["cluster_label"] = cluster_df["hidden_change_category"]

# Cluster-level summary.
if cluster_df.empty:
    cluster_summary = pd.DataFrame()
else:
    cluster_summary = (
        cluster_df.groupby(
            ["cluster", "cluster_label", "hidden_change_category"], dropna=False
        )
        .agg(
            records=("title", "count"),
            source_count=("source", "nunique"),
            avg_hidden_change_score=("hidden_change_score", "mean"),
            avg_seed_score=("business_idea_seed_score", "mean"),
            max_seed_score=("business_idea_seed_score", "max"),
            latest_event=("event_date", "max"),
        )
        .reset_index()
        .sort_values(["avg_seed_score", "source_count", "records"], ascending=False)
    )
    cluster_summary[["avg_hidden_change_score", "avg_seed_score", "max_seed_score"]] = (
        cluster_summary[
            ["avg_hidden_change_score", "avg_seed_score", "max_seed_score"]
        ].round(2)
    )

cluster_summary.head(20)

## 7. Convert hidden changes into business idea hypotheses

This is a deterministic idea-generation layer. It creates practical startup/service/product ideas from the evidence clusters.

You can later replace this with an LLM summarizer, but this version is reproducible and does not send data to an external model.

In [ ]:
IDEA_ARCHETYPES = {
    "regulatory_friction_shift": {
        "archetype": "Compliance intelligence + review workflow",
        "target": "agencies and brands that publish ads, influencer content, testimonials, AI-generated creative, or regulated claims",
        "solution": "monitor new rules and enforcement, map them to active campaigns, and generate review checklists before launch",
        "mvp": "a weekly FTC/NAD/platform-policy digest plus a campaign claim-review checklist generator",
        "pricing": "subscription per agency seat or per client portfolio",
    },
    "measurement_data_shift": {
        "archetype": "Privacy-safe measurement and attribution product",
        "target": "agencies losing attribution accuracy because of cookies, privacy rules, and platform tracking limits",
        "solution": "combine first-party data, server-side events, incrementality tests, and MMM into a practical agency workflow",
        "mvp": "a lightweight audit that scores a client's tracking stack and recommends the next measurement experiments",
        "pricing": "monthly analytics retainer, SaaS dashboard, or implementation package",
    },
    "ai_capability_shift": {
        "archetype": "AI operating system for agency delivery",
        "target": "agency teams trying to use AI without damaging brand voice, compliance, QA, or client trust",
        "solution": "turn briefs into controlled creative/workflow agents with brand rules, human review, and audit logs",
        "mvp": "brief-to-draft assistant with brand voice rules, claims checklist, and approval workflow",
        "pricing": "seat-based SaaS plus implementation services",
    },
    "platform_rule_distribution_shift": {
        "archetype": "Platform-change monitoring and remediation service",
        "target": "paid media teams dependent on Google, Meta, TikTok, Amazon, LinkedIn, retail media, and programmatic platforms",
        "solution": "detect policy/API/algorithm changes, predict affected accounts, and produce migration playbooks",
        "mvp": "weekly platform-change dashboard with account impact checklist",
        "pricing": "subscription per media team or managed service retainer",
    },
    "client_budget_procurement_shift": {
        "archetype": "Agency ROI proof and retention system",
        "target": "agencies facing fee pressure, procurement scrutiny, churn, or client in-housing",
        "solution": "package evidence of incremental lift, operational savings, and strategic value into renewal-ready client narratives",
        "mvp": "client-retention report generator using campaign, CRM, and activity data",
        "pricing": "agency operations SaaS or retention consulting package",
    },
    "competitive_structure_shift": {
        "archetype": "Vertical wedge agency or competitive intelligence product",
        "target": "independent agencies competing against holding companies, consultancies, platforms, and client in-housing",
        "solution": "identify underserved verticals/workflows where large agencies are slow and self-serve tools are weak",
        "mvp": "niche opportunity report plus competitor/client-positioning database",
        "pricing": "subscription intelligence product or productized strategy sprint",
    },
    "new_tooling_supply_shift": {
        "archetype": "Package open-source/dev tools into agency-ready workflows",
        "target": "non-technical agency teams that cannot operationalize raw GitHub/API tooling",
        "solution": "wrap emerging open-source/adtech/martech tools with setup, templates, dashboards, and support",
        "mvp": "one-click deployable toolkit for tracking, reporting, QA, or creative automation",
        "pricing": "hosted SaaS, support subscription, or implementation fee",
    },
    "legal_trust_reputation_shift": {
        "archetype": "Trust, claims, and brand-safety risk layer",
        "target": "agencies and brands exposed to deceptive claims, fake reviews, AI content, creator risk, or brand-safety failures",
        "solution": "screen campaign assets and creator partnerships for legal/reputation risk before publishing",
        "mvp": "risk checklist and reviewer queue for claims, reviews, influencers, and synthetic content",
        "pricing": "per-review usage pricing or compliance workflow subscription",
    },
    "talent_workflow_shift": {
        "archetype": "Agency workforce transformation toolkit",
        "target": "agency leaders managing AI reskilling, utilization pressure, QA burden, and changing delivery workflows",
        "solution": "measure workflows, find automation candidates, train teams, and track utilization/quality outcomes",
        "mvp": "workflow audit plus AI playbook and utilization dashboard",
        "pricing": "training package plus recurring analytics software",
    },
    "macro_ad_spend_shift": {
        "archetype": "Budget-risk early warning and scenario planner",
        "target": "agency owners and CMOs planning spend during macro uncertainty",
        "solution": "monitor ad-spend indicators, customer demand signals, and industry-specific budget risk",
        "mvp": "monthly spend-risk dashboard and scenario templates",
        "pricing": "subscription intelligence report or CFO/CMO planning add-on",
    },
    "general_shift": {
        "archetype": "Industry-change intelligence product",
        "target": "operators who need early warning before obvious market shifts",
        "solution": "aggregate weak signals and convert them into practical strategy and product experiments",
        "mvp": "weekly hidden-change report with validation backlog",
        "pricing": "subscription newsletter, research portal, or consulting retainer",
    },
}


def short_title(text: str, max_words: int = 8) -> str:
    words = re.findall(r"[A-Za-z0-9][A-Za-z0-9\-]+", str(text))
    if not words:
        return "hidden industry shift"
    return " ".join(words[:max_words])


def md_link(title: str, url: str) -> str:
    title = compact_spaces(title) or "Untitled"
    if url:
        return f"[{title}]({url})"
    return title


def evidence_for_group(group: pd.DataFrame, n: int = 5) -> list[dict[str, str]]:
    out = []
    for _, row in (
        group.sort_values(
            ["business_idea_seed_score", "hidden_change_score"], ascending=False
        )
        .head(n)
        .iterrows()
    ):
        event_date = row.get("event_date")
        if pd.notna(event_date):
            event_date = pd.to_datetime(event_date).date().isoformat()
        else:
            event_date = "unknown"
        out.append(
            {
                "source": str(row.get("source", "unknown")),
                "date": event_date,
                "title": str(row.get("title", "Untitled")),
                "url": str(row.get("url", "")),
                "hidden_change_score": str(row.get("hidden_change_score", "")),
            }
        )
    return out


def build_idea_row(cluster_key: tuple[Any, ...], group: pd.DataFrame) -> dict[str, Any]:
    cluster, cluster_label, category = cluster_key
    category = category or "general_shift"
    archetype = IDEA_ARCHETYPES.get(category, IDEA_ARCHETYPES["general_shift"])
    sources = sorted(group["source"].dropna().astype(str).unique().tolist())
    source_count = len(sources)
    records = len(group)
    avg_hidden = float(group["hidden_change_score"].mean()) if records else 0.0
    avg_seed = float(group["business_idea_seed_score"].mean()) if records else 0.0
    latest = group["event_date"].max() if "event_date" in group else pd.NaT
    latest_str = (
        pd.to_datetime(latest).date().isoformat() if pd.notna(latest) else "unknown"
    )

    terms = []
    for col in [
        "hidden_trigger_terms",
        "pain_terms",
        "opportunity_terms",
        "matched_industry_terms",
    ]:
        for vals in group.get(col, []):
            if isinstance(vals, list):
                terms.extend(vals)
    top_terms = [
        term for term, _ in Counter([str(t).lower() for t in terms]).most_common(10)
    ]
    keyword_label = (
        cluster_label
        if cluster_label and str(cluster_label) != "nan"
        else simple_keywords(group["text"].tolist())
    )

    diversity_bonus = min(source_count / 5, 1.0)
    evidence_bonus = min(records / 12, 1.0)
    avg_score_component = (avg_seed + avg_hidden) / 200
    idea_score = round(
        100
        * (0.55 * avg_score_component + 0.25 * diversity_bonus + 0.20 * evidence_bonus),
        2,
    )

    idea_title = f"{archetype['archetype']}: {short_title(keyword_label)}"
    hidden_change = (
        f"Signals suggest a {category.replace('_', ' ')} around: {keyword_label}. "
        f"Top repeated terms: {', '.join(top_terms[:8]) if top_terms else 'none detected'}."
    )
    business_idea = f"Build a {archetype['archetype'].lower()} for {archetype['target']} that can {archetype['solution']}."
    why_hidden = (
        f"The theme appears across {source_count} source type(s): {', '.join(sources[:8])}. "
        f"Several high-hiddenness sources such as filings, policy sources, research, developer forums, or niche sites may surface this before mainstream business media."
    )
    why_now = (
        f"Latest signal date: {latest_str}. Average hidden-change score: {avg_hidden:.1f}. "
        f"Average idea seed score: {avg_seed:.1f}."
    )
    wedge = archetype["mvp"]
    validation = (
        "Interview 10 target users; ask what changed in the last 6 months, what workflow broke, "
        "what they already tried, and whether they would pay for the proposed MVP. Then build a no-code/manual concierge version for 2 pilot customers."
    )
    moat = (
        "Accumulate proprietary labeled signals, customer workflow data, benchmarks, and post-change playbooks. "
        "Over time this becomes an industry-specific early-warning and operating dataset."
    )
    risk = (
        "May be too broad or noisy. Validate urgency and willingness to pay before building software. "
        "For compliance/legal ideas, involve qualified counsel before making legal claims."
    )

    evidence = evidence_for_group(group, n=5)
    evidence_md = "\n".join(
        [
            f"- {e['source']} | {e['date']} | {md_link(e['title'], e['url'])}"
            for e in evidence
        ]
    )

    return {
        "idea_score": idea_score,
        "idea_title": idea_title,
        "hidden_change_category": category,
        "cluster": cluster,
        "cluster_label": keyword_label,
        "records": records,
        "source_count": source_count,
        "sources": "; ".join(sources),
        "latest_signal_date": latest_str,
        "avg_hidden_change_score": round(avg_hidden, 2),
        "avg_seed_score": round(avg_seed, 2),
        "target_customer": archetype["target"],
        "hidden_change": hidden_change,
        "business_idea": business_idea,
        "why_hidden": why_hidden,
        "why_now": why_now,
        "mvp_wedge": wedge,
        "monetization": archetype["pricing"],
        "validation_experiment": validation,
        "possible_moat": moat,
        "main_risk": risk,
        "evidence_markdown": evidence_md,
    }


if cluster_df.empty:
    ideas_df = pd.DataFrame()
else:
    idea_rows = []
    for key, group in cluster_df.groupby(
        ["cluster", "cluster_label", "hidden_change_category"], dropna=False
    ):
        # Require at least some industry relevance or a strong hidden-change score.
        if len(group) < 1:
            continue
        if (
            group["business_idea_seed_score"].max() < 20
            and group["hidden_change_score"].max() < 20
        ):
            continue
        idea_rows.append(build_idea_row(key, group))
    ideas_df = (
        pd.DataFrame(idea_rows)
        .sort_values(["idea_score", "avg_seed_score"], ascending=False)
        .reset_index(drop=True)
    )

ideas_df.head(15)

## 8. Create a validation backlog

A business idea is only useful if you can test it. This cell turns each idea into a concrete validation task.

In [ ]:
def validation_tasks_for_idea(row: pd.Series) -> list[dict[str, Any]]:
    title = row.get("idea_title", "Untitled idea")
    customer = row.get("target_customer", "target users")
    return [
        {
            "idea_title": title,
            "priority": "P0",
            "task_type": "customer_discovery",
            "task": f"Find 10 people in this segment: {customer}. Ask what changed recently, what is painful, and what they pay for today.",
            "success_criteria": "At least 4/10 report the pain as urgent and name a current workaround or budget owner.",
        },
        {
            "idea_title": title,
            "priority": "P0",
            "task_type": "evidence_review",
            "task": "Manually review the top evidence links and remove false positives.",
            "success_criteria": "At least 5 strong evidence items remain after manual review.",
        },
        {
            "idea_title": title,
            "priority": "P1",
            "task_type": "concierge_mvp",
            "task": "Offer a manual version of the workflow to 2 pilot customers before writing production code.",
            "success_criteria": "One customer agrees to a paid pilot or repeated weekly use.",
        },
        {
            "idea_title": title,
            "priority": "P1",
            "task_type": "pricing_test",
            "task": "Test three price points: low subscription, premium subscription, and done-for-you service retainer.",
            "success_criteria": "Prospects do not reject the middle price immediately and ask implementation questions.",
        },
    ]


if ideas_df.empty:
    validation_backlog = pd.DataFrame()
else:
    tasks = []
    for _, row in ideas_df.head(20).iterrows():
        tasks.extend(validation_tasks_for_idea(row))
    validation_backlog = pd.DataFrame(tasks)

validation_backlog.head(20)

## 9. Visualize hidden changes and idea scores

In [ ]:
import matplotlib.pyplot as plt

if not cluster_df.empty:
    category_counts = (
        cluster_df["hidden_change_category"].value_counts().sort_values(ascending=True)
    )
    ax = category_counts.plot(
        kind="barh", figsize=(10, max(4, len(category_counts) * 0.45))
    )
    ax.set_title(f"Hidden-change categories detected: {INDUSTRY}")
    ax.set_xlabel("Record count")
    ax.set_ylabel("Category")
    plt.tight_layout()
    plt.show()
else:
    print("No cluster data to plot.")

if not ideas_df.empty:
    plot_df = ideas_df.head(15).sort_values("idea_score", ascending=True)
    ax = plot_df.plot(
        kind="barh",
        x="idea_title",
        y="idea_score",
        legend=False,
        figsize=(12, max(5, len(plot_df) * 0.5)),
    )
    ax.set_title("Top business idea hypotheses")
    ax.set_xlabel("Idea score")
    ax.set_ylabel("")
    plt.tight_layout()
    plt.show()
else:
    print("No ideas to plot.")

## 10. Export CSVs and Markdown report

In [ ]:
# Prepare CSV-safe exports.
cluster_export = cluster_df.copy()
for col in [
    "matched_industry_terms",
    "hidden_trigger_terms",
    "pain_terms",
    "opportunity_terms",
    "hidden_change_categories",
]:
    if col in cluster_export.columns:
        cluster_export[col] = cluster_export[col].map(
            lambda x: "; ".join(x) if isinstance(x, list) else x
        )
if "hidden_change_matches" in cluster_export.columns:
    cluster_export["hidden_change_matches"] = cluster_export[
        "hidden_change_matches"
    ].map(lambda x: json.dumps(x, ensure_ascii=False) if isinstance(x, dict) else x)

export_cols = [
    "source",
    "event_date",
    "published_date",
    "fetched_at",
    "record_type",
    "hidden_change_category",
    "hidden_change_categories",
    "cluster",
    "cluster_label",
    "hidden_change_score",
    "business_idea_seed_score",
    "relevance_score",
    "change_language_score",
    "source_hiddenness",
    "freshness_score",
    "cross_domain_score",
    "title",
    "abstract",
    "url",
    "matched_industry_terms",
    "hidden_trigger_terms",
    "pain_terms",
    "opportunity_terms",
]
for col in export_cols:
    if col not in cluster_export.columns:
        cluster_export[col] = None

cluster_export[export_cols].to_csv(CLEAN_CSV, index=False)
cluster_summary.to_csv(CLUSTERS_CSV, index=False)
ideas_df.to_csv(IDEAS_CSV, index=False)
validation_backlog.to_csv(VALIDATION_BACKLOG_CSV, index=False)


def report_idea(row: pd.Series, rank: int) -> str:
    return "\n".join(
        [
            f"## {rank}. {row.get('idea_title', 'Untitled idea')}",
            "",
            f"**Idea score:** {row.get('idea_score', 0)}",
            f"**Hidden-change category:** `{row.get('hidden_change_category', '')}`",
            f"**Target customer:** {row.get('target_customer', '')}",
            "",
            f"**Hidden change:** {row.get('hidden_change', '')}",
            "",
            f"**Business idea:** {row.get('business_idea', '')}",
            "",
            f"**Why this may be hidden:** {row.get('why_hidden', '')}",
            "",
            f"**Why now:** {row.get('why_now', '')}",
            "",
            f"**MVP wedge:** {row.get('mvp_wedge', '')}",
            "",
            f"**Monetization:** {row.get('monetization', '')}",
            "",
            f"**Validation experiment:** {row.get('validation_experiment', '')}",
            "",
            f"**Possible moat:** {row.get('possible_moat', '')}",
            "",
            f"**Main risk:** {row.get('main_risk', '')}",
            "",
            "**Evidence:**",
            row.get("evidence_markdown", "No evidence captured."),
            "",
        ]
    )


report_parts = [
    f"# Hidden-Change Business Ideas Report: {INDUSTRY}",
    "",
    f"Generated: {datetime.now().isoformat(timespec='seconds')}",
    "",
    f"Industry description: {INDUSTRY_DESCRIPTION}",
    "",
    f"Window: {START_DATE} to {END_DATE}",
    "",
    f"Records analyzed after dedupe: {len(cluster_df):,}",
    f"Business ideas generated: {len(ideas_df):,}",
    "",
    "## How to read this report",
    "",
    "- Hidden-change score favors relevance, change-trigger language, source hiddenness, freshness, and cross-domain overlap.",
    "- Idea score favors clusters with high seed scores, multiple source types, and enough evidence.",
    "- This is an opportunity discovery tool, not proof that a business will work.",
    "- Validate ideas manually with customer interviews and paid pilots before building software.",
    "",
]

if not cluster_summary.empty:
    report_parts += ["## Hidden-change cluster summary", ""]
    for _, row in cluster_summary.head(20).iterrows():
        report_parts.append(
            f"- **{row.get('cluster_label')}** / `{row.get('hidden_change_category')}`: "
            f"{int(row.get('records', 0))} records, {int(row.get('source_count', 0))} sources, "
            f"avg seed score {row.get('avg_seed_score')}"
        )
    report_parts.append("")

if ideas_df.empty:
    report_parts += [
        "## Business idea hypotheses",
        "",
        "No ideas generated. Widen the date window, add API keys, or loosen industry terms.",
        "",
    ]
else:
    report_parts += ["## Business idea hypotheses", ""]
    for rank, (_, row) in enumerate(ideas_df.head(25).iterrows(), start=1):
        report_parts.append(report_idea(row, rank))

report_parts += [
    "## Validation backlog",
    "",
]
if validation_backlog.empty:
    report_parts.append("No validation tasks generated.")
else:
    for _, task in validation_backlog.head(60).iterrows():
        report_parts.append(
            f"- **{task.get('priority')} / {task.get('task_type')}** — {task.get('task')} "
            f"Success criteria: {task.get('success_criteria')}"
        )

REPORT_MD.write_text("\n".join(report_parts), encoding="utf-8")

print(f"Wrote {CLEAN_CSV}")
print(f"Wrote {CLUSTERS_CSV}")
print(f"Wrote {IDEAS_CSV}")
print(f"Wrote {VALIDATION_BACKLOG_CSV}")
print(f"Wrote {REPORT_MD}")

## 11. Optional: schedule as a daily idea-discovery job

Once the notebook works manually, schedule it with `papermill`:

```bash
pip install papermill
papermill hidden_change_business_ideas_all_apis.ipynb runs/hidden_change_business_ideas_$(date +%F).ipynb
```

Recommended production loop:

1. Run daily or weekly.
2. Store outputs by date.
3. Compare new clusters against old clusters.
4. Alert only when a cluster is new, growing, or crossing a score threshold.
5. Keep manually labeled outcomes: false positive, interesting, validated, paid pilot, killed.

## 12. How to turn this into a real product

A stronger product version would add:

- persistent Postgres tables for records, clusters, ideas, and validations
- pgvector embeddings for semantic dedupe and deeper retrieval
- Redis cache for hot dashboard reads
- an MCP tool layer: `search_signals`, `get_idea_evidence`, `log_validation`, `trigger_action`
- confirmation-gated write-backs to Notion, Google Sheets, Slack, or GitHub Issues
- human labels for “good idea,” “false positive,” “urgent,” and “willingness to pay”
- source-specific connectors for industry newsletters, client emails, CRM notes, and call transcripts

The moat is not just collecting public data. The moat is the labeled map between **hidden change → buyer pain → validated business idea → outcome**.